# ECG Arrhythmia Classification — Exploration & Training

MIT-BIH Arrhythmia dataset (Kaggle "Heartbeat" CSV version), 1D CNN in PyTorch.

**This notebook is for exploration.** The production code lives in `src/`. The notebook re-uses those modules where possible so there is no hidden state — anything important here also runs from a plain `python -m src.train`.

Sections:
1. Load the data
2. Inspect shapes / missing values / class distribution
3. Plot sample heartbeats per class
4. What the labels mean
5. Prepare data (already normalised — we just check)
6. Train (calls `src.train`)
7. Evaluate (calls `src.evaluate`)

In [ ]:
# Make the project root importable so `from src import ...` works from notebooks/.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config

print("Project root:", PROJECT_ROOT)
print("Train CSV exists:", config.TRAIN_CSV.exists())
print("Test  CSV exists:", config.TEST_CSV.exists())

## 1. Load the data

The Kaggle CSVs have **no header row**, so `header=None`. There are 188 columns: 187 signal samples + 1 label in the last column.

In [ ]:
train_df = pd.read_csv(config.TRAIN_CSV, header=None)
test_df = pd.read_csv(config.TEST_CSV, header=None)

print("Train shape:", train_df.shape)
print("Test  shape:", test_df.shape)
train_df.head()

## 2. Shapes, missing values, class distribution

In [ ]:
# Split into signals (X) and labels (y). Last column is the label.
X_train = train_df.iloc[:, :-1].values.astype(np.float32)
y_train = train_df.iloc[:, -1].values.astype(int)
X_test = test_df.iloc[:, :-1].values.astype(np.float32)
y_test = test_df.iloc[:, -1].values.astype(int)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("Signal length:", X_train.shape[1], "(expect 187)")
print("Missing values in train:", np.isnan(X_train).sum())
print("Missing values in test :", np.isnan(X_test).sum())
print("Signal min/max:", X_train.min(), X_train.max(), "(already ~[0,1])")

In [ ]:
# Class distribution — note the heavy imbalance toward class 0 (Normal).
unique, counts = np.unique(y_train, return_counts=True)
dist = pd.DataFrame({
    "label": unique,
    "name": [config.CLASS_NAMES[i] for i in unique],
    "description": [config.CLASS_DESCRIPTIONS[i] for i in unique],
    "count": counts,
    "percent": (counts / counts.sum() * 100).round(2),
})
dist

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar([config.CLASS_NAMES[i] for i in unique], counts, color="steelblue")
plt.yscale("log")  # log scale because class 0 dwarfs the rest
plt.ylabel("count (log scale)")
plt.xlabel("class")
plt.title("Training set class distribution (note the imbalance)")
plt.show()

## 3. Plot a sample heartbeat from each class

Each row is one heartbeat aligned around its R-peak. Different arrhythmia types have visibly different morphologies.

In [ ]:
fig, axes = plt.subplots(1, config.NUM_CLASSES, figsize=(18, 3), sharey=True)
for cls in range(config.NUM_CLASSES):
    idx = np.where(y_train == cls)[0][0]  # first beat of this class
    axes[cls].plot(X_train[idx])
    axes[cls].set_title(f"{cls}: {config.CLASS_NAMES[cls]}")
    axes[cls].set_xlabel("sample")
axes[0].set_ylabel("amplitude")
plt.suptitle("One example heartbeat per class")
plt.tight_layout()
plt.show()

## 4. What the labels mean

The 5 classes follow the AAMI grouping of the original MIT-BIH annotations:

| Label | Symbol | Meaning |
|-------|--------|---------|
| 0 | N | Normal beat |
| 1 | S | Supraventricular ectopic beat |
| 2 | V | Ventricular ectopic beat |
| 3 | F | Fusion beat (fusion of normal + ventricular) |
| 4 | Q | Unknown / unclassifiable / paced beat |

Clinically, the **minority** classes (S, V, F) are the interesting ones — missing a ventricular beat matters more than misclassifying a normal one. That is why we report **per-class recall and macro-F1**, not just accuracy.

## 5. Prepare data

The signals are already amplitude-normalised to ~[0, 1] and zero-padded to a fixed length of 187, so **no extra scaling is required**. Our `ECGDataset` just adds a channel dimension to make the shape `(1, 187)` for `nn.Conv1d`.

In [ ]:
from src.dataset import build_dataloaders

train_loader, val_loader, test_loader, class_weights = build_dataloaders()
xb, yb = next(iter(train_loader))
print("Batch signals:", tuple(xb.shape), "(expect (B, 1, 187))")
print("Batch labels :", tuple(yb.shape))
print("Class weights:", class_weights.numpy().round(3))

## 6. Train

We just call the production training loop. It trains the 1D CNN, selects the best epoch by validation macro-F1, and saves to `models/ecg_cnn.pt`.

> Tip: edit hyperparameters in `src/config.py` (epochs, batch size, learning rate, imbalance strategy).

In [ ]:
from src.train import train

best_metrics = train()
best_metrics

## 7. Evaluate on the test set

Generates the classification report + confusion matrix and saves them to `results/`.

In [ ]:
from src.evaluate import evaluate

evaluate()

In [ ]:
# Show the saved confusion matrix inline.
from IPython.display import Image
Image(filename=str(config.CONFUSION_MATRIX_PATH))